In [ ]:
# Import packages
from pathlib import Path
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
# Paths for unemployment rate and credit spread datasets
REPO_ROOT = Path(__file__).resolve().parent.parent    
UNRATE_CSV = REPO_ROOT / "Datasets" / "alfred_unrate.csv"
CREDIT_CSV = REPO_ROOT / "Datasets" / "alfred_moodys_baa_10y_spread.csv"
LOANS_CSV = REPO_ROOT / "Datasets" / "LC_loans.csv"
OUT = Path(__file__).resolve().parent

In [ ]:
# Snapshot of unemployment rate dataset
employment_df = pd.read_csv(UNRATE_CSV)
employment_df.head()

,observation_date,UNRATE_20260605,UNRATE_20260702
0,2006-12-01,4.4,4.4
1,2007-01-01,4.6,4.6
2,2007-02-01,4.5,4.5
3,2007-03-01,4.4,4.4
4,2007-04-01,4.5,4.5


In [251]:
# Data types
employment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  235 non-null    object 
 1   UNRATE_20260605   233 non-null    float64
 2   UNRATE_20260702   234 non-null    float64
dtypes: float64(2), object(1)
memory usage: 5.6+ KB


In [252]:
# Are the UNRATE_20260605 and UNRATE_20260702 columns ever different?
filter = (employment_df['UNRATE_20260605'] == employment_df['UNRATE_20260702'])
employment_df[filter].count()

observation_date    233
UNRATE_20260605     233
UNRATE_20260702     233
dtype: int64

In [253]:
# Return rows for which UNRATE_20260605 and UNRATE_20260702 are different
not_filter = filter = (employment_df['UNRATE_20260605'] != employment_df['UNRATE_20260702'])
employment_df[not_filter]

,observation_date,UNRATE_20260605,UNRATE_20260702
226,2025-10-01,NaN,NaN
234,2026-06-01,NaN,4.2


In [254]:
# Clean dataset
employment_df['observation_date'] = pd.to_datetime(employment_df['observation_date'])
employment_df['year'] = employment_df['observation_date'].dt.year
employment_df['month'] = employment_df['observation_date'].dt.month
employment_df.drop(columns=['observation_date', 'UNRATE_20260702'], inplace=True)
employment_df.dropna(inplace=True)
employment_df.rename(columns={'UNRATE_20260605':'unemployment_rate'}, inplace=True)
employment_df.head()

,unemployment_rate,year,month
0,4.4,2006,12
1,4.6,2007,1
2,4.5,2007,2
3,4.4,2007,3
4,4.5,2007,4


In [255]:
# Lag unemployment rate by one month to compensate for lagged data reporting
num_rows = employment_df.shape[0]
employment_df['unemployment_rate'] = employment_df['unemployment_rate'].shift(1)
employment_df.drop([0], inplace=True)
employment_df.reset_index(drop=True, inplace=True)
employment_df.head()

,unemployment_rate,year,month
0,4.4,2007,1
1,4.6,2007,2
2,4.5,2007,3
3,4.4,2007,4
4,4.5,2007,5


In [ ]:
# Import credit spreads
credit_spread_df = pd.read_csv(CREDIT_CSV)
credit_spread_df.head()

,observation_date,BAA10Y
0,2007-01-02,1.64
1,2007-01-03,1.61
2,2007-01-04,1.62
3,2007-01-05,1.60
4,2007-01-08,1.59


In [257]:
# Clean dataset
credit_spread_df['observation_date'] = pd.to_datetime(credit_spread_df['observation_date'])
credit_spread_df['year'] = credit_spread_df['observation_date'].dt.year
credit_spread_df['month'] = credit_spread_df['observation_date'].dt.month
credit_spread_df['day'] = credit_spread_df['observation_date'].dt.day
credit_spread_df.drop(columns=['observation_date'], inplace=True)
credit_spread_df.rename(columns={'BAA10Y':'credit_spread'}, inplace=True)
credit_spread_df.head()

,credit_spread,year,month,day
0,1.64,2007,1,2
1,1.61,2007,1,3
2,1.62,2007,1,4
3,1.60,2007,1,5
4,1.59,2007,1,8


In [258]:
# Replace credit spreads with monthly averages
credit_spread_df = credit_spread_df.groupby(by=['year','month']).mean()
credit_spread_df.drop(columns=['day'], inplace=True)
credit_spread_df.head()


credit_spread
year month               
2007 1           1.583333
     2           1.553158
     3           1.703182
     4           1.699524
     5           1.639091

In [ ]:
# Prepare LC_loans for joining with employment and credit data
LC_loans_df = pd.read_csv(LOANS_CSV)
LC_loans_df['issue_d'] = pd.to_datetime(LC_loans_df['issue_d'], format='%b-%Y')
LC_loans_df['year'] = LC_loans_df['issue_d'].dt.year
LC_loans_df['month'] = LC_loans_df['issue_d'].dt.month
LC_loans_df.drop(columns=['issue_d'], inplace=True)
LC_loans_df.head()

/tmp/ipykernel_59741/3766642357.py:2: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  LC_loans_df = pd.read_csv('LC_loans.csv')


,id,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc,year,month
0,68407277,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,190xx,0,Debt consolidation,NaN,2015,12
1,68355089,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,577xx,0,Business,NaN,2015,12
2,68341763,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,605xx,0,NaN,NaN,2015,12
3,68476807,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,174xx,0,Major purchase,NaN,2015,12
4,68426831,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,300xx,0,Debt consolidation,NaN,2015,12


In [260]:
# Joining LC_loans with employment and credit data
LC_loans_df = LC_loans_df.merge(employment_df, on=['year', 'month'], how='left')
LC_loans_df = LC_loans_df.merge(credit_spread_df, on=['year', 'month'], how='left')
LC_loans_df.head()

,id,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc,year,month,unemployment_rate,credit_spread
0,68407277,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,190xx,0,Debt consolidation,NaN,2015,12,5.1,3.212273
1,68355089,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,577xx,0,Business,NaN,2015,12,5.1,3.212273
2,68341763,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,605xx,0,NaN,NaN,2015,12,5.1,3.212273
3,68476807,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,174xx,0,Major purchase,NaN,2015,12,5.1,3.212273
4,68426831,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,300xx,0,Debt consolidation,NaN,2015,12,5.1,3.212273


In [261]:
# Restore original issue_d column as datetime
LC_loans_df['issue_d'] = pd.to_datetime(
    LC_loans_df[['year', 'month']].assign(day=1)
).dt.strftime('%b-%Y')
LC_loans_df.drop(columns=['year', 'month'], inplace=True)
LC_loans_df.head()

,id,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,Default,title,desc,unemployment_rate,credit_spread,issue_d
0,68407277,55000.0,5.91,3600,677.0,1,10+ years,debt_consolidation,MORTGAGE,PA,190xx,0,Debt consolidation,NaN,5.1,3.212273,Dec-2015
1,68355089,65000.0,16.06,24700,717.0,1,10+ years,small_business,MORTGAGE,SD,577xx,0,Business,NaN,5.1,3.212273,Dec-2015
2,68341763,71000.0,13.85,20000,697.0,1,10+ years,home_improvement,MORTGAGE,IL,605xx,0,NaN,NaN,5.1,3.212273,Dec-2015
3,68476807,104433.0,25.37,10400,697.0,1,3 years,major_purchase,MORTGAGE,PA,174xx,0,Major purchase,NaN,5.1,3.212273,Dec-2015
4,68426831,34000.0,10.20,11950,692.0,1,4 years,debt_consolidation,RENT,GA,300xx,0,Debt consolidation,NaN,5.1,3.212273,Dec-2015


In [262]:
# Check dataset for nulls in columns
LC_loans_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1347681 entries, 0 to 1347680
Data columns (total 17 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   id                 1347681 non-null  int64  
 1   revenue            1347681 non-null  float64
 2   dti_n              1347681 non-null  float64
 3   loan_amnt          1347681 non-null  int64  
 4   fico_n             1347681 non-null  float64
 5   experience_c       1347681 non-null  int64  
 6   emp_length         1347681 non-null  object 
 7   purpose            1347681 non-null  object 
 8   home_ownership_n   1347681 non-null  object 
 9   addr_state         1347681 non-null  object 
 10  zip_code           1347680 non-null  object 
 11  Default            1347681 non-null  int64  
 12  title              1331024 non-null  object 
 13  desc               119099 non-null   object 
 14  unemployment_rate  1347681 non-null  float64
 15  credit_spread      1347681 non-n

In [263]:
# Change NaN values in zip_code, title and desc to "Unknown" for model building
LC_loans_df.fillna(value='Unknown', inplace=True)

In [ ]:
# Write output CSV
LC_loans_df.to_csv(f'{OUT}/LC_loans_unrate_credit.csv', index=False)